# Imports

In [1]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [4]:
api.start_spark(n_executors=400, config=config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


22/12/01 10:04:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
22/12/01 10:04:18 WARN DomainSocketFactory: The short-circuit local reads feature cannot be used because libhadoop cannot be loaded.
22/12/01 10:04:48 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Attempted to request executors before the AM has registered!


# Set Parameters

Set date range and airports here.

In [5]:
year0 = "2019"

In [6]:
year1 = str(int(year0) + 1)
#year1 = year0

In [7]:
dates = {"start_date": year0 + "-10-01", "end_date": year1 + "-01-01"}

In [8]:
airports = [
    "KADW",
    "KATL",
    "KBOS",
    "KBWI",
    "KCLT",
    "KDCA",
    "KDEN",
    "KDFW",
    "KDTW",
    "KEWR",
    "KFLL",
    "KIAD",
    "KIAH",
    "KJFK",
    "KLAS",
    "KLAX",
    "KLGA",
    "KMCO",
    "KMDW",
    "KMEM",
    "KMIA",
    "KMSP",
    "KORD",
    "KPHL",
    "KPHX",
    "KSAN",
    "KSDF",
    "KSEA",
    "KSFO",
    "KSLC",
    "KTPA",
    "PANC",
    "PHNL",
]

In [9]:
#airports = [ "KATL", ]

# df_aa_raw

In [10]:
dates

{'start_date': '2019-10-01', 'end_date': '2020-01-01'}

In [12]:
df_aa_raw = (
    api.dataframe("AirspaceAssignment", **dates, partition_filters=[
        api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "SECTOR"]),
        api.custom_partitions(["ASSOCIATED", "StaticEramAirspace", "TRACON"])
        ], metadata=True)
)

# get the schema for the entry and exit point

Define the schema of the litetrack points structure array

In [13]:
df_aa_point = (df_aa_raw.select("entry_point"))

In [14]:
aa_point_json = df_aa_point.schema.json()

In [17]:
aa_point_schema = T.StructType.fromJson(json.loads(aa_point_json))

# define functions

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

functions to stringify the airspace assignment row and the entry/exit point columns

In [ ]:
def strinigfy_aa_point(aa_point:aa_point_schema) -> str:
    output = ""
    output += str(aa_point.time) + " "
    output += f'{aa_point.latitude:.6f}' + " "
    output += f'{aa_point.longitude:.6f}' + " "
    output += f'{aa_point.altitude:.0f}'
    return output

strinigfy_aa_point_udf = F.udf(strinigfy_aa_point, T.StringType())

In [ ]:
def strinigfy_aa(artcc:T.StringType, airspace_identifier:T.StringType, airspace_class:T.StringType, airspace_subclass:T.StringType, entry_point:aa_point_schema, exit_point:aa_point_schema, duration:T.DoubleType, distance:T.DoubleType) -> str:
    output = ""
    output += artcc + " "
    output += "'" + airspace_identifier + "' "
#    output += airspace_class + " "
    output += airspace_subclass + " "
    output += strinigfy_aa_point(entry_point) + " "
    output += strinigfy_aa_point(exit_point) + " "
    output += f'{duration:.1f}' + " "
    output += f'{distance:.1f}'
    return output

strinigfy_aa_udf = F.udf(strinigfy_aa, T.StringType())

# df_aa

sort the df_aa_raw data, create an "artcc" column from the meta_custom_partitions[1],  and stringify the needed data to an "airspace_assignment" column

In [ ]:
df_aa = (
    df_aa_raw
    .sort(F.col("track_key"), F.col("entry_point.time"))
    .withColumn("artcc", df_aa_raw.metadata.custom_partitions[1])
    .withColumn("airspace_assignment", strinigfy_aa_udf("artcc", "airspace_identifier", "airspace_class", "airspace_subclass", "entry_point", "exit_point", "known_visit_duration", "known_distance_covered"))
    .select(
        "track_key",
        "airspace_assignment",
    )
)

# df_aa_grouped

group by track_key and concatenate "airspace_assignment"

In [ ]:
df_aa_grouped = (
    df_aa.groupBy("track_key").agg(F.concat_ws(":", F.collect_list("airspace_assignment")).alias("airspace_assignments"))
)

# df_fps

get flightplans for the airports of interest

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [ ]:
@F.udf("string")
def to_date(ts):
    return datetime.datetime.utcfromtimestamp(ts / 1000).strftime("%Y%m%d")

In [ ]:
df_fps = (
    api.dataframe("FlightplanSeries", **dates, partition_filters=api.custom_partitions("ASSOCIATED"), metadata=True)
    .select(
        "track_key",
        to_date("metadata.effective_start_date").alias("start_date"),
        to_date("metadata.effective_end_date").alias("end_date"),
        "callsign",
        "aircraft_type",
        "mode_s_code",
        F.col("initial_departure_aerodrome").alias("orig"),
        F.col("final_destination_aerodrome").alias("dest"),
    )
    .withColumn("month", F.substring("start_date", 5, 2))
    .filter(F.col("orig") != F.col("dest"))
    .filter(F.col("orig").isin(airports) | F.col("dest").isin(airports))
)

# df_fps_combined

create the arrivals and departures, then combine them

In [ ]:
df_arrivals = (
    df_fps.withColumn("airport", F.col("dest"))
    .withColumn("operation", F.lit("ARRIVALS"))
    .filter(F.col("airport").isin(airports))
)

In [ ]:
df_departures = (
    df_fps.withColumn("airport", F.col("orig"))
    .withColumn("operation", F.lit("DEPARTURES"))
    .filter(F.col("airport").isin(airports))
)

In [ ]:
df_fps_combined = df_arrivals.unionByName(df_departures)

# df_aa_joined

filter aa based on the track_keys in the flightplans for the cities we care about

In [ ]:
df_aa_joined = (
    df_aa_grouped
    .join(df_fps_combined, on=["track_key"], how="inner")
)

# output the results

In [ ]:
df_aa_output = (
    df_aa_joined
    .select(
        "track_key",
        "start_date",
        "end_date",
        "orig",
        "dest",
        "callsign",
        "mode_s_code",
        "aircraft_type",
        "airspace_assignments",
        "airport",
        "month",
        "operation",
    )
)

In [ ]:
(
    df_aa_output
    .repartition("airport", "operation", "month")
    .write.option("header", True).partitionBy(["airport", "operation", "month"])
#    .csv("CRAFT/" + year0 + "/airspace_assignments", compression="gzip", mode="overwrite")
    .csv("CRAFT/" + dates["start_date"] + "/airspace_assignments", compression="gzip", mode="overwrite")
)